# Oil Palm Disease Classification using ALOS-2 SAR

**Postgraduate Research** — Prerequisites & Environment Setup

This notebook sets up the full environment for processing ALOS-2 PALSAR-2 data on **Google Colab**:
- **ESA SNAP 13.0.0** (Sentinel Toolboxes) — SAR processing engine
- **esa_snappy** — Python bridge to SNAP's Java API
- **Scientific Python stack** — numpy, rasterio, scikit-learn, etc.

---

## Prerequisites

### 0. Mount Google Drive & Set Project Root

Mount your Drive so the SNAP installation and data persist across Colab sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = "/content/drive/MyDrive/SAR-OilPalm"

import os
os.makedirs(f"{PROJECT_ROOT}/data", exist_ok=True)
os.makedirs(f"{PROJECT_ROOT}/snap", exist_ok=True)
os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")
print(f"Working directory: {os.getcwd()}")

### 1. System Setup — Python Environment + Dependencies

Installs `uv` (fast package manager), creates a virtual environment on Drive (skips if already exists), and installs all Python packages.

In [ ]:
# System checks
import platform, sys, shutil, os
print(f"Platform: {platform.system()} {platform.release()}")
print(f"Architecture: {platform.machine()}")
print(f"Python: {sys.version}")
print(f"wget: {shutil.which('wget') or 'not found'}")
print(f"curl: {shutil.which('curl') or 'not found'}")
print(f"java: {shutil.which('java') or 'not found (SNAP bundles its own JRE)'}")

In [ ]:
%%bash
set -e

PROJECT_ROOT="/content/drive/MyDrive/SAR-OilPalm"
cd "$PROJECT_ROOT"

# Install uv if missing
if ! command -v uv &>/dev/null; then
    echo "Installing uv..."
    curl -LsSf https://astral.sh/uv/install.sh | sh
    source "$HOME/.cargo/env"
fi
echo "uv: $(uv --version)"

# Create virtual environment (skip if already exists)
if [ ! -d ".venv" ]; then
    echo "Creating virtual environment..."
    uv venv --python 3.12 .venv
else
    echo "Virtual environment already exists, skipping creation"
fi
source .venv/bin/activate

# Install core Python packages (idempotent)
uv pip install \
    esa-snappy \
    numpy>=1.26 scipy \
    matplotlib seaborn \
    rasterio rioxarray \
    geopandas shapely pyproj \
    scikit-learn \
    jupyter ipykernel ipywidgets \
    tqdm joblib

echo ""
echo "=== Environment ready ==="
echo "Python: $(.venv/bin/python --version)"

### 2. Download & Install ESA SNAP 13.0.0

Downloads the Sentinel Toolboxes installer (~1 GB) and runs a quiet headless installation to `PROJECT_ROOT/snap`.
**Note:** SNAP is installed on Google Drive so it persists across Colab sessions. If already installed, the download is skipped.

In [ ]:
%%bash
set -e

PROJECT_ROOT="/content/drive/MyDrive/SAR-OilPalm"
cd "$PROJECT_ROOT"

INSTALLER="esa-snap_sentinel_linux-13.0.0.sh"
INSTALLER_URL="https://download.esa.int/step/snap/13.0/installers/${INSTALLER}"
SNAP_HOME="$PROJECT_ROOT/snap"

# Check if SNAP is already installed
if [ -f "$SNAP_HOME/bin/snappy-conf" ]; then
    echo "SNAP already installed at $SNAP_HOME, skipping download and install"
    exit 0
fi

# Download if not already present
if [ ! -f "$INSTALLER" ]; then
    echo "Downloading SNAP 13.0.0 Sentinel Toolboxes (~1 GB)..."
    wget "$INSTALLER_URL" -O "$INSTALLER"
fi
chmod +x "$INSTALLER"
echo "Installer ready: $(du -h $INSTALLER | cut -f1)"

# Create response file for quiet headless install
cat > response.varfile << 'EOF'
sys.adminRights$Boolean=false
sys.component.RSTB$Boolean=true
sys.component.S1TBX$Boolean=true
sys.component.S2TBX$Boolean=true
sys.component.S3TBX$Boolean=false
sys.component.SNAP$Boolean=true
sys.installationDir=__SNAP_HOME__
sys.languageId=en
sys.programGroupDisabled$Boolean=true
createDesktopLinkAction$Boolean=false
executeLauncherWithPythonAction$Boolean=false
deleteSnapDir$Boolean=false
EOF

# Inject actual path into response file
sed -i "s|__SNAP_HOME__|$SNAP_HOME|" response.varfile

# Run installer in quiet mode
mkdir -p "$SNAP_HOME"
echo "Installing SNAP to $SNAP_HOME (this may take a few minutes)..."
bash "$INSTALLER" -q -varfile response.varfile
echo ""
echo "=== SNAP installed at $SNAP_HOME ==="
ls -la "$SNAP_HOME/bin/" | head -5

### 3. Configure esa_snappy (SNAP ↔ Python Bridge)

Connects SNAP's Java engine to Python via `esa_snappy`, then verifies everything works.

In [ ]:
%%bash
set -e

PROJECT_ROOT="/content/drive/MyDrive/SAR-OilPalm"
SNAP_HOME="$PROJECT_ROOT/snap"
VENV_PYTHON="$PROJECT_ROOT/.venv/bin/python"

export PATH="$SNAP_HOME/bin:$PATH"

echo "=== Installing esa_snappy ==="
"$VENV_PYTHON" -m pip install esa-snappy

echo ""
echo "=== Running snappy-conf ==="
"$SNAP_HOME/bin/snappy-conf" "$VENV_PYTHON"

echo ""
echo "=== Verification ==="
"$VENV_PYTHON" -c "
import esa_snappy
from esa_snappy import ProductIO, GPF
print(f'esa_snappy version: {esa_snappy.__version__}')
print(f'SNAP home: {esa_snappy.get_snap_home()}')
print('OK - All imports successful')
"

echo ""
echo "=== GPT CLI ==="
"$SNAP_HOME/bin/gpt" -h 2>&1 | head -8 || true

# Bump JVM memory to 8G (70% of typical 12GB RAM)
SNAPPY_INI=$("$VENV_PYTHON" -c "import esa_snappy; import pathlib; print(pathlib.Path(esa_snappy.__file__).parent / 'esa_snappy.ini')")
if [ -f "$SNAPPY_INI" ]; then
    echo ""
    echo "=== Memory settings ==="
    grep -q 'java_max_mem' "$SNAPPY_INI" && \
        sed -i 's/^# *java_max_mem:.*/java_max_mem: 8G/' "$SNAPPY_INI" || \
        echo "java_max_mem: 8G" >> "$SNAPPY_INI"
    echo "JVM max memory set to 8G"
fi

echo ""
echo "=== Setup Complete ==="

### 4. Quick Test — Load a Product

Test the Python-SNAP bridge by reading a sample product.

In [ ]:
%%bash
source /content/drive/MyDrive/SAR-OilPalm/.venv/bin/activate

python -c "
from esa_snappy import ProductIO, GPF
print('esa_snappy is ready for ALOS-2 SAR processing')
print('Available operators:', len(GPF.getDefaultInstance().getOperatorSpiRegistry().getOperatorSpis()))
"

---
**Next steps:**
1. Place your ALOS-2 PALSAR-2 scenes in `PROJECT_ROOT/data/`
2. Re-run only **Cell 0** (Drive mount) and **Cell 1** (env setup) on new Colab sessions — SNAP will be reused from Drive
3. Proceed to the preprocessing section to calibrate, speckle-filter, and terrain-correct the imagery